*Imports & Basic Paths*

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from glob import glob
import os
import yaml
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd

DATA_ROOT = Path(r"D:\Dataset")
YAML_PATH = DATA_ROOT / "plantdoc.yaml"
WEIGHTS_PATH = DATA_ROOT / "yolov8n.pt"

print("DATA_ROOT:", DATA_ROOT)
print("YAML_PATH:", YAML_PATH)
print("WEIGHTS_PATH:", WEIGHTS_PATH)

print("Train images dir:", DATA_ROOT / "train" / "images")
print("Test images dir:", DATA_ROOT / "test" / "images")

*Detect Number of Classes*

In [ ]:
labels_root_train = DATA_ROOT / "train" / "labels"
labels_root_test = DATA_ROOT / "test" / "labels"

label_files = glob(str(labels_root_train / "*.txt")) + \
              glob(str(labels_root_test / "*.txt"))

print("Total label files found:", len(label_files))
print("First few label files:")
for lf in label_files[:5]:
    print("  ", lf)

cls_ids = set()

for lf in label_files:
    with open(lf, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            try:
                cls = int(parts[0])
                cls_ids.add(cls)
            except ValueError:
                pass

print("\nUnique class ids:", sorted(cls_ids))

if not cls_ids:
    raise RuntimeError("No Class ID Found!")

max_cls = max(cls_ids)
nc = max_cls + 1
print("Max class id:", max_cls)
print("=> nc should be:", nc)

*Update plantdoc.yaml*

In [ ]:
cfg = {
    "path": "D:/Dataset",
    "train": "train/images",
    "val": "test/images",
    "test": "test/images",
    "nc": int(nc),
    "names": [f"class_{i}" for i in range(int(nc))],
}

print("Saving YAML to:", YAML_PATH)

with open(YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)

print("✅ plantdoc.yaml written.")
print(cfg)

*Check Paths & Load Pretrained Model*

In [ ]:
with open(YAML_PATH, "r", encoding="utf-8") as f:
    cfg_loaded = yaml.safe_load(f)

print("Loaded from YAML:")
print(cfg_loaded)

root = cfg_loaded["path"]
train_dir = os.path.join(root, cfg_loaded["train"])
val_dir = os.path.join(root, cfg_loaded["val"])

print("\nTrain dir:", train_dir, "exists:", os.path.exists(train_dir))
print("Val dir:", val_dir, "exists:", os.path.exists(val_dir))

assert WEIGHTS_PATH.exists(), f"Weights file not found: {WEIGHTS_PATH}"

model = YOLO(str(WEIGHTS_PATH))
print("\n✅ Pretrained model loaded:", WEIGHTS_PATH)


*Run Models*

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml
import os

DATA_ROOT = Path(r"D:\Dataset")
YAML_PATH = DATA_ROOT / "plantdoc.yaml"
WEIGHTS_PATH = DATA_ROOT / "yolov8n.pt"

model = YOLO(str(WEIGHTS_PATH))

results = model.train(
    data=str(YAML_PATH),
    epochs=400,
    imgsz=416,
    batch=8,
    device=0,
    amp=False,
    workers=0,
    close_mosaic=0,
    lr0=1e-3,
    weight_decay=5e-4,
    project=str(DATA_ROOT / "runs_final2"),
    name="yolov8n_plantdoc",
    pretrained=True,
    patience=10,
    verbose=True,
)

print("✅ Training finished.")

*Functions for GT vs Pred Visualization*

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from glob import glob
import os
import yaml
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd

%matplotlib inline

DATA_ROOT = Path(r"D:\Dataset")
YAML_PATH = DATA_ROOT / "plantdoc.yaml"
BEST_WEIGHTS = DATA_ROOT / "runs_final2" / "yolov8n_plantdoc" / "weights" / "best.pt"

print("DATA_ROOT:", DATA_ROOT)
print("YAML_PATH:", YAML_PATH)
print("BEST_WEIGHTS:", BEST_WEIGHTS)

assert YAML_PATH.exists(), f"YAML not found: {YAML_PATH}"
assert BEST_WEIGHTS.exists(), f"best.pt not found: {BEST_WEIGHTS}"

# YAML → گرفتن نام کلاس‌ها
with open(YAML_PATH, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg.get("names", None)
print("nc:", data_cfg.get("nc"))
print("len(names):", len(class_names) if class_names else None)

best_model = YOLO(str(BEST_WEIGHTS))
print("✅ Best model loaded.")

*Plot All Losses & Metrics from results.csv*

In [ ]:
run_dir = DATA_ROOT / "runs_final2" / "yolov8n_plantdoc"
results_csv = run_dir / "results.csv"

print("Results path:", results_csv)
assert results_csv.exists(), f"results.csv not found at: {results_csv}"

df = pd.read_csv(results_csv)
print("Columns:")
print(df.columns.tolist())

epochs = df["epoch"]

train_loss_cols = [c for c in df.columns if c.startswith("train/") and "loss" in c]
val_loss_cols   = [c for c in df.columns if c.startswith("val/")   and "loss" in c]

print("\nTrain loss columns:", train_loss_cols)
print("Val loss columns:", val_loss_cols)

plt.figure(figsize=(10, 6))
for col in train_loss_cols:
    plt.plot(epochs, df[col], label=col)

for col in val_loss_cols:
    plt.plot(epochs, df[col], linestyle="--", label=col)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Losses")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

metric_cols = [c for c in df.columns if c.startswith("metrics/")]
print("\nMetric columns:", metric_cols)

plt.figure(figsize=(10, 6))
for col in metric_cols:
    plt.plot(epochs, df[col], label=col)

plt.xlabel("Epoch")
plt.ylabel("Metric value")
plt.title("Validation Metrics (precision, recall, mAP)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()